# Notebook 01 of 04: Fit one adsorption isotherm

**Goal.** Start from raw experimental data (`Ce`, `qe`) and fit the 13 isotherm models used in the manuscript.

**Use the built-in example first.** Leave `DATA_FILE = "sample_isotherm_data.csv"` and run the notebook from top to bottom. After that works, replace `DATA_FILE` with your own CSV or Excel file.

**Your input file must contain only the essentials at first:**

| Column | Meaning | Typical unit |
|---|---|---|
| `Ce` | equilibrium concentration after adsorption | mg/L |
| `qe` | adsorbed amount at equilibrium | mg/g |

**What you should look at.** The most useful outputs are the fit overlay, residual plot, fitted-parameter table, metric table, and descriptor table containing `Qmax`, `Kaff_star`, `Eads`, and `sigma_H`.

**Stop condition.** Continue to Notebook 02 only after at least one physically reasonable model converges and the overlay does not show a systematic miss.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import pandas as pd

from utils_isotherms import (
    MODELS,
    compute_SSE, compute_R2, compute_AIC, compute_BIC,
    compute_EABS, compute_RESID, compute_ARED, compute_MPSED, compute_HYBRID,
    extract_descriptors,
)

In [ ]:
def build_fit_config(Ce, q_exp):
    """
    Build data-adaptive initial guesses and bounds for all 13 isotherm models.
    Scales p0 values from the experimental data so fitting converges regardless
    of the concentration / capacity range.
    """
    _Qg  = float(np.max(q_exp)) * 1.5        # generous Qmax starting guess
    _Kg  = 1.0 / float(np.median(Ce))         # affinity scale from data
    _Qhi = float(np.max(q_exp)) * 20          # upper bound for Qmax

    return {
        'Langmuir':         {'p0': [_Qg, _Kg],
                             'bounds': ([0, 0], [_Qhi, np.inf])},
        'Double Langmuir':  {'p0': [_Qg * 0.6, _Kg * 2, _Qg * 0.4, _Kg * 0.3],
                             'bounds': ([0, 0, 0, 0], [_Qhi, np.inf, _Qhi, np.inf])},
        'Freundlich':       {'p0': [max(_Qg * _Kg**0.5, 1e-3), 0.5],
                             'bounds': ([0, 0.05], [np.inf, 2.0])},
        'Sips':             {'p0': [_Qg, _Kg, 0.8],
                             'bounds': ([0, 0, 0.1], [_Qhi, np.inf, 2.0])},
        'Toth':             {'p0': [_Qg, _Kg, 0.8],
                             'bounds': ([0, 0, 0.1], [_Qhi, np.inf, 2.0])},
        'Jovanovic':        {'p0': [_Qg, _Kg],
                             'bounds': ([0, 0], [_Qhi, np.inf])},
        'Temkin':           {'p0': [_Kg, 1.0],
                             'bounds': ([0, 1e-6], [np.inf, np.inf])},
        'UNILAN':           {'p0': [_Qg, _Kg * 5, _Kg / 5],
                             'bounds': ([0, 1e-10, 1e-10], [_Qhi, np.inf, np.inf])},
        'Redlich-Peterson': {'p0': [_Qg * _Kg, _Kg, 0.9],
                             'bounds': ([0, 0, 0.01], [np.inf, np.inf, 1.0])},
        'Koble-Corrigan':   {'p0': [max(_Qg * _Kg**0.5, 1e-3), max(_Kg**0.5, 1e-6), 0.5],
                             'bounds': ([0, 0, 0.05], [np.inf, np.inf, 2.0])},
        'Radke-Prausnitz':  {'p0': [_Qg, _Qg * _Kg, 0.5],
                             'bounds': ([0, 0, 0.01], [np.inf, np.inf, 1.0])},
        'D-R/D-A':          {'p0': [_Qg, 8.0, 2.0],
                             'bounds': ([0, 0.1, 1.0], [_Qhi, 50.0, 6.0])},
        'Hill':             {'p0': [_Qg, 1.0 / _Kg, 1.0],
                             'bounds': ([0, 0, 0.1], [_Qhi, np.inf, 4.0])},
    }

In [ ]:
print("Notebook 01/04: fitting raw Ce-qe data")
print("Example input: sample_isotherm_data.csv")
print("Change DATA_FILE only after the example runs successfully.")


## Load experimental data

Expected columns: `Ce`, `qe`. The loader also accepts `ce`/`qe` (case-insensitive).


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                        DATA FILE CONFIGURATION                               ║
# ║                                                                              ║
# ║  Set DATA_FILE below to the name of your experimental isotherm data file.    ║
# ║  Place the file in the SAME FOLDER as this notebook.                         ║
# ║                                                                              ║
# ║  File requirements:                                                          ║
# ║    - Format  : CSV (.csv) or Excel (.xlsx / .xls)                            ║
# ║    - Columns : Ce  (equilibrium concentration, mg/L)                         ║
# ║                qe  (adsorption capacity, mg/g)                               ║
# ║    - Column names are case-insensitive (Ce, ce, CE all work).                ║
# ║                                                                              ║
# ║  Examples:                                                                   ║
# ║    DATA_FILE = "sample_isotherm_data.csv"          # included sample         ║
# ║    DATA_FILE = "my_experiment.xlsx"                 # your own data          ║
# ║                                                                              ║
# ║  A sample file (sample_isotherm_data.csv) is included for testing.           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

DATA_FILE = "sample_isotherm_data.csv"   # <--- CHANGE THIS to your data file

# ══════════════════════════════════════════════════════════════════════════════
#  Do NOT modify anything below this line unless you know what you are doing.
# ══════════════════════════════════════════════════════════════════════════════

# ── Resolve path relative to this notebook's folder ───────────────────────────
_NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
DATA_PATH = os.path.join(_NOTEBOOK_DIR, DATA_FILE)

# ── Validate input ────────────────────────────────────────────────────────────
if not DATA_FILE or DATA_FILE.strip() == "":
    raise ValueError(
        "\n\n"
        "  *** ERROR: No data file specified! ***\n\n"
        "  Go to the top of this cell and set DATA_FILE to your file name.\n\n"
        "  Example:\n"
        "      DATA_FILE = 'sample_isotherm_data.csv'\n"
    )

if not os.path.isfile(DATA_PATH):
    raise FileNotFoundError(
        f"\n\n"
        f"  *** ERROR: File not found: '{DATA_FILE}' ***\n\n"
        f"  Looked in: {_NOTEBOOK_DIR}\n\n"
        f"  Place your data file in the same folder as this notebook.\n"
    )

# ── Load data ─────────────────────────────────────────────────────────────────
ext = os.path.splitext(DATA_FILE)[1].lower()
if ext == '.csv':
    df_data = pd.read_csv(DATA_PATH)
elif ext in ('.xlsx', '.xls'):
    df_data = pd.read_excel(DATA_PATH)
else:
    raise ValueError(
        f"\n  Unsupported file format '{ext}'. Use .csv or .xlsx\n"
    )

# Normalise column names (case-insensitive)
col_map = {c.strip().lower(): c for c in df_data.columns}
if 'ce' not in col_map or 'qe' not in col_map:
    raise ValueError(
        f"\n\n"
        f"  *** ERROR: Required columns not found! ***\n\n"
        f"  Your file has columns: {list(df_data.columns)}\n"
        f"  This notebook expects two columns named 'Ce' and 'qe'\n"
        f"  (case-insensitive: Ce, ce, CE, etc. all work).\n"
    )

df_data = df_data.rename(columns={col_map['ce']: 'Ce', col_map['qe']: 'qe'})
df_data = df_data[['Ce', 'qe']].dropna()
df_data = df_data[(df_data['Ce'] > 0) & (df_data['qe'] >= 0)].sort_values('Ce')

if len(df_data) < 4:
    raise ValueError(
        f"\n\n"
        f"  *** ERROR: Not enough valid data points ({len(df_data)})! ***\n\n"
        f"  At least 4 data points are needed for meaningful fitting.\n"
        f"  Check your file for missing or negative values.\n"
    )

Ce = df_data['Ce'].to_numpy(dtype=float)
q_exp = df_data['qe'].to_numpy(dtype=float)

print(f"Loaded {len(df_data)} data points from '{DATA_FILE}'")
print(f"  Ce range : {Ce.min():.4g} -- {Ce.max():.4g} mg/L")
print(f"  qe range : {q_exp.min():.4g} -- {q_exp.max():.4g} mg/g")

## Fit candidate isotherm models


In [ ]:
# All 13 isotherm models from main.tex / SI.tex
model_names = [
    'Langmuir', 'Double Langmuir', 'Freundlich', 'Sips', 'Toth',
    'Jovanovic', 'Temkin', 'UNILAN', 'Redlich-Peterson',
    'Koble-Corrigan', 'Radke-Prausnitz', 'D-R/D-A', 'Hill',
]

# Build data-adaptive initial guesses and bounds
fit_config = build_fit_config(Ce, q_exp)

results = []
n = len(Ce)

for model_name in model_names:
    func, param_names, p = MODELS[model_name]
    cfg = fit_config[model_name]
    try:
        with np.errstate(over='ignore', invalid='ignore', divide='ignore'):
            popt, pcov = curve_fit(
                func, Ce, q_exp,
                p0=cfg['p0'], bounds=cfg['bounds'], maxfev=30000,
            )
            q_pred = func(Ce, *popt)
        sse   = compute_SSE(q_exp, q_pred)
        r2    = compute_R2(q_exp, q_pred)
        aic   = compute_AIC(n, sse, p)
        bic   = compute_BIC(n, sse, p)
        eabs  = compute_EABS(q_exp, q_pred)
        resid = compute_RESID(q_exp, q_pred)
        ared  = compute_ARED(q_exp, q_pred)
        mpsed = compute_MPSED(q_exp, q_pred)
        hybrid = compute_HYBRID(q_exp, q_pred, p)
        results.append({
            'model': model_name, 'params': popt, 'param_names': param_names,
            'n_params': p, 'func': func, 'q_pred': q_pred,
            'SSE': sse, 'R2': r2, 'AIC': aic, 'BIC': bic,
            'EABS': eabs, 'RESID': resid, 'ARED': ared,
            'MPSED': mpsed, 'HYBRID': hybrid,
        })
        print(f"  ✓ {model_name:20s}  R²={r2:.6f}  SSE={sse:.4f}")
    except Exception as e:
        results.append({
            'model': model_name, 'params': None, 'param_names': param_names,
            'n_params': p, 'func': func, 'q_pred': None,
            'SSE': np.nan, 'R2': np.nan, 'AIC': np.nan, 'BIC': np.nan,
            'EABS': np.nan, 'RESID': np.nan, 'ARED': np.nan,
            'MPSED': np.nan, 'HYBRID': np.nan, 'error': str(e),
        })
        print(f"  ✗ {model_name:20s}  FAILED — {e}")

# Summary table sorted by AIC
summary = pd.DataFrame([{
    'Model': r['model'], 'p': r['n_params'],
    'SSE': r['SSE'], 'R²': r['R2'],
    'AIC': r['AIC'], 'BIC': r['BIC'],
    'EABS': r['EABS'], 'RESID': r['RESID'],
    'ARED': r['ARED'], 'MPSED': r['MPSED'], 'HYBRID': r['HYBRID'],
} for r in results])
summary.sort_values('AIC', inplace=True)
print(f"\n{'='*70}")
print(f"  {len([r for r in results if r['params'] is not None])}/{len(model_names)} models converged")
print(f"{'='*70}")
summary

## Plot experimental data and top model fits


In [ ]:
# Plot all converged model fits
fig, ax = plt.subplots(figsize=(10, 6))

# Experimental data
ax.scatter(Ce, q_exp, color='black', label='Experimental', s=60,
           zorder=15, edgecolors='white', linewidths=0.5)

# Smooth evaluation grid (slightly beyond data range)
Ce_fine = np.linspace(Ce.min() * 0.9, Ce.max() * 1.05, 300)

# Sort by SSE (best first) for consistent legend ordering
ok_results = sorted([r for r in results if r['q_pred'] is not None],
                    key=lambda x: x['SSE'])
n_ok = len(ok_results)
cmap = plt.cm.tab20

for i, r in enumerate(ok_results):
    try:
        with np.errstate(over='ignore', invalid='ignore', divide='ignore'):
            q_fine = r['func'](Ce_fine, *r['params'])
        ax.plot(Ce_fine, q_fine,
                color=cmap(i / max(n_ok - 1, 1)),
                linewidth=2, alpha=0.85,
                label=f"{r['model']} (R²={r['R2']:.4f})")
    except Exception:
        pass

ax.set_xlabel('Ce (mg L⁻¹)', fontsize=12)
ax.set_ylabel('qe (mg g⁻¹)', fontsize=12)
ax.set_title(f'All Isotherm Model Fits — {os.path.basename(DATA_FILE)}', fontsize=13)
ax.legend(fontsize=8, loc='best', ncol=2)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## Fitted parameters for each converged model

In [ ]:
# Display fitted parameters for each model
rows = []
for r in results:
    if r['params'] is None:
        rows.append({'Model': r['model'], 'Status': 'FAILED'})
        continue
    row = {'Model': r['model'], 'Status': 'OK'}
    for name, val in zip(r['param_names'], r['params']):
        row[name] = val
    rows.append(row)

df_params = pd.DataFrame(rows).set_index('Model')
df_params

## Universal descriptor extraction

Extract {Q_max, ln K*_aff, E_ads, σ_H} fingerprint from each converged model.

In [ ]:
# Extract universal descriptors for each converged model
desc_rows = []
for r in results:
    if r['params'] is None:
        continue
    # Build param dict for extract_descriptors
    param_dict = dict(zip(r['param_names'], r['params']))
    try:
        d = extract_descriptors(r['model'], param_dict)
        desc_rows.append({
            'Model':       r['model'],
            'Q_max':       d['Qmax'],
            'ln_Kaff':     np.log(d['Kaff_star']) if d['Kaff_star'] > 0 else np.nan,
            'E_ads (kJ/mol)': d['Eads'],
            'σ_H (kJ/mol)':   d['sigma_H'],
        })
    except Exception as e:
        desc_rows.append({
            'Model': r['model'], 'Q_max': np.nan,
            'ln_Kaff': np.nan, 'E_ads (kJ/mol)': np.nan,
            'σ_H (kJ/mol)': np.nan,
        })

df_desc = pd.DataFrame(desc_rows).set_index('Model')
print("Universal descriptors (C⁰ = 1 mg/L, T = 298 K)")
df_desc

## Notebook 01 checkpoint

Before moving on, write down three things in your lab notes or copy the output table:

1. Which model has the smallest residual error?
2. Does the fitted curve follow the low-concentration and high-concentration regions?
3. Are the extracted descriptors finite and physically meaningful (`Qmax`, `Eads`, `sigma_H`)?

If the answer to item 2 or 3 is no, do not rank materials yet. Fix the input data, units, or model bounds first.
